In [2]:
!pip install tqdm
!gdown https://drive.google.com/drive/folders/1c4QsSigwAXGJM4M6ZXlt1_Ebg_Qmhiwy?usp=sharing --fuzzy

Downloading...
From: https://drive.google.com/drive/folders/1c4QsSigwAXGJM4M6ZXlt1_Ebg_Qmhiwy?usp=sharing
To: /content/1c4QsSigwAXGJM4M6ZXlt1_Ebg_Qmhiwy?usp=sharing
1.30MB [00:00, 27.5MB/s]


In [3]:
!ls -l /content/

total 1272
-rw-r--r-- 1 root root 1297202 Nov 23 14:42 '1c4QsSigwAXGJM4M6ZXlt1_Ebg_Qmhiwy?usp=sharing'
drwxr-xr-x 1 root root    4096 Nov 20 14:30  sample_data


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import shutil
import random
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

Usando o dispositivo: cpu


In [7]:
data_dir = '/content/drive/MyDrive/PlantVillage' ###ALTERAR PARA O LOCAL DO DATASET PLANTVILLAGE
train_dir = './data/train'
test_dir = './data/test'

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Imagens por classe
class_names = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
class_sample_count = []
for class_name in class_names:
    class_dir = os.path.join(data_dir, class_name)
    images = [img for img in os.listdir(class_dir) if img.endswith(('.jpg', '.png', '.jpeg', '.JPG'))]
    class_sample_count.append(len(images))

    train_class_dir = os.path.join(train_dir, class_name)
    test_class_dir = os.path.join(test_dir, class_name)
    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(test_class_dir, exist_ok=True)

    random.shuffle(images)
    split_index = int(len(images) * 0.8)
    train_images = images[:split_index]
    test_images = images[split_index:]

    for img in train_images:
        shutil.copy(os.path.join(class_dir, img), os.path.join(train_class_dir, img))
    for img in test_images:
        shutil.copy(os.path.join(class_dir, img), os.path.join(test_class_dir, img))

    print(f"Class {class_name}: {len(train_images)} train images, {len(test_images)} test images")

print("Dataset split completed!")


Class Tomato__Tomato_YellowLeaf__Curl_Virus: 2566 train images, 642 test images
Class Tomato__Target_Spot: 1123 train images, 281 test images
Class Tomato__Tomato_mosaic_virus: 298 train images, 75 test images
Class Tomato_Spider_mites_Two_spotted_spider_mite: 1340 train images, 336 test images
Class Tomato_Late_blight: 1527 train images, 382 test images
Class Tomato_healthy: 1272 train images, 319 test images
Class Tomato_Septoria_leaf_spot: 1416 train images, 355 test images
Class Tomato_Early_blight: 800 train images, 200 test images
Class Tomato_Leaf_Mold: 761 train images, 191 test images
Class Tomato_Bacterial_spot: 1701 train images, 426 test images
Class Potato___Late_blight: 800 train images, 200 test images
Class Potato___Early_blight: 800 train images, 200 test images
Class Pepper__bell___healthy: 1182 train images, 296 test images
Class Pepper__bell___Bacterial_spot: 797 train images, 200 test images
Class Potato___healthy: 121 train images, 31 test images
Dataset split com

In [8]:
# 1. Data augmentation
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(15), #Rotações aleatórias
    transforms.RandomHorizontalFlip(), #Flips aleatórios
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), #Muda brilho, contraste, saturação e tom das imagens aleatoriamente
    transforms.ToTensor(), #Transforma em tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) #pesos comuns
])

# 2. Testes. Sem aumento
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [9]:
class VGG16(nn.Module):
    def __init__(self, num_classes):
        super(VGG16, self).__init__()
        self.features = nn.Sequential(
            # Convolução 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1), #de 3 pra 64 canais
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),      #Max pool reduz a dimensão espacial para 64x64

            # Convolução 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1), #de 64 pra 128 canais
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),        #64 pra 32x32

            # Convolução 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),#de 128 pra 256 canais
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),       #16x16

            # Convolução 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1), #512 canais
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),         #8x8

            # Convolução 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1), #Mantém 512
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),         #4x4
        )

#=====================CLASSIFIER=====================#

        self.classifier = nn.Sequential(
            nn.Linear(512 * 4 * 4, 4096), #FC
            nn.ReLU(inplace=True),
            nn.Dropout(0.5), #Camada de dropout que desliga 50% dos neurônios
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes), #saída com o número de classes, 15
        )
        # Inicializando pesos
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x) #Passa os dados pelas camadas convolucionais
        x = x.view(x.size(0), 512 * 4 * 4) #flatten
        x = self.classifier(x)#classifica
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # Funções de inicialização de pesos
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d): #batcch normalization
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                # Xavier pras camadas densas
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

In [10]:
num_classes = len(train_dataset.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #teste pra definir se GPU ou CPU
model = VGG16(num_classes).to(device) #Definição do modelo


train_labels = [label for _, label in train_dataset] #Extraindo os rotos e balanceando o peso de cada um, quanto menos tem mais ele vale
classes = np.unique(train_labels)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to(device) #np -> tensor

criterion = nn.CrossEntropyLoss(weight=class_weights) #loss function
# gradiente descendente estocástico
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
# decay de peso (L2), por Isso o AdamW e não o Adam
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(model)

VGG16(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation

In [ ]:
epochs = 20
train_losses = []
test_accuracies = []
all_true_labels = []
all_pred_labels = []
patience = 3
min_delta = 0.001
best_loss = float('inf')
epochs_no_improve = 0

for epoch in range(epochs): #Loop de treinamento
    model.train()
    running_loss = 0.0
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Treinamento:]")#Barrinha de treinamento bonitinha
    for images, labels in train_progress:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        train_progress.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    test_epoch_loss = epoch_loss
    if test_epoch_loss < best_loss - min_delta:
        best_loss = test_epoch_loss
        epochs_no_improve = 0

        torch.save(model.state_dict(), 'best_model_checkpoint.pth')
        print(f"Salvando o melhor modelo na época {epoch + 1} com Loss: {best_loss:.4f}")
    else:
        epochs_no_improve += 1
        print(f"Nenhuma melhoria. Contador de paciência: {epochs_no_improve}/{patience}")
    if epochs_no_improve >= patience:
      print(f"!!! Early Stopping acionado na Época {epoch + 1} !!!")
      break # Sai do laço 'for epoch'
    # Learning rate muda a cada epoch
    scheduler.step()

    # Fase de evaluation
    model.eval()
    correct = 0
    total = 0
    test_progress = tqdm(test_loader, desc=f"Epoch {epoch + 1}/{epochs} [Testando]")
    with torch.no_grad():
        for images, labels in test_progress:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            all_true_labels.extend(labels.cpu().numpy())
            all_pred_labels.extend(predicted.cpu().numpy())

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_accuracy = 100 * correct / total
    test_accuracies.append(epoch_accuracy)

    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%, LR: {current_lr:.6f}")


Epoch 1/20 [Treinamento:]:   2%|▏         | 12/516 [05:01<3:45:22, 26.83s/it, loss=2.69]

In [ ]:
model.eval() #Modo de avaliação (desliga o dropout e afins)
with torch.no_grad():
    # Carrega só um batch
    images, labels = next(iter(test_loader))
    # iterador
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# Figura pra mostrar 5 imgs
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    # Tem que mandar de volta pra cpu se não não plota
    img = images[i].cpu().permute(1, 2, 0).numpy()
    img = img * 0.5 + 0.5
    axes[i].imshow(np.clip(img, 0, 1))


    pred_class = train_dataset.classes[predicted[i]]
    true_class = train_dataset.classes[labels[i]]


    axes[i].set_title(f"Pred: {pred_class}\nTrue: {true_class}",
                     color="green" if pred_class == true_class else "red")
    axes[i].axis('off')
plt.show()

In [ ]:
# Plotando o grafico da função de loss
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, marker='o', color='red')
plt.title('Loss por epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)

# Plotando o gráfico de evolução da acurácia ao longo das epochs
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), test_accuracies, marker='s', color='green')
plt.title('Acurácia por epoch')
plt.xlabel('Epoch')
plt.ylabel('Accurácia (%)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------- Matrizz de confusão --------------------------
print("\n=== Matriz de Confusão ===")
# 1. Calculando matriz
cm = confusion_matrix(all_true_labels, all_pred_labels)
# 2. Normalizaz matriz
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# 3. Matriz de confusão como heatmap
plt.figure(figsize=(18, 15))
sns.heatmap(
    cm_normalized,
    annot=False,
    cmap='PuBu',
    xticklabels=train_dataset.classes,  # x Previsões
    yticklabels=train_dataset.classes   # y Verdadeiros
)
plt.title('Normalized Confusion Matrix (Plant Disease VGG16)', fontsize=16)
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


print(f"Forma da matriz: {cm.shape} (Num de classes: {num_classes})")
print("Distribuição de Previsão para as primeiras 5 classes (Classe Verdadeira → Classe Mais Frequente Predita):")
for i in range(min(5, num_classes)):
    true_class = train_dataset.classes[i]
    top_pred_idx = cm[i].argmax()
    top_pred_class = train_dataset.classes[top_pred_idx]
    print(f"  {true_class} Mais predito: {top_pred_class} (contagem: {cm[i][top_pred_idx]})")
# -------------------------------------------------------------------------

# -------------------------- Salvar modelo --------------------------
rint("\n=== Modelo Salvo no Diretório de Trabalho ===")
# 1. Definição do Caminho de Salvamento
# Removemos a pasta save_dir. O diretório de trabalho atual é representado por '.'
# Não é necessário criar subdiretórios, o arquivo será salvo diretamente.

final_accuracy = test_accuracies[-1] # Obtém a acurácia da última rodada
# Define o caminho de salvamento: O arquivo será salvo diretamente no diretório atual (./)
save_path = f"alexnet_plant_disease_acc{final_accuracy:.2f}%.pth" # <-- ALTERAÇÃO AQUI!

# 2. Salvamento (Gravação no Disco)
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': train_dataset.classes,
    'num_classes': num_classes,
    'input_size': (3, 128, 128),
    'optimizer_state_dict': optimizer.state_dict(),
    'final_accuracy': final_accuracy
}, save_path)

# 3. Verificação do Salvamento
if os.path.exists(save_path):
    print(f"Modelo salvo com sucesso! Arquivo: {save_path}")

    # Demonstração de carregamento (Verifica a integridade)
    checkpoint = torch.load(save_path)
    print("Verificação de Carregamento - Informações Chave Salvas:")
    print(f"  - Final Accuracy: {checkpoint['final_accuracy']:.2f}%")
else:
    print("Falha ao salvar o modelo!")